# Vérification du téléchargement des réseaux techniques

Ce notebook vérifie que `download.reseaux.download_reseaux` interroge correctement les couches de réseaux techniques (BD TOPO IGN : lignes électriques, postes de transformation, pylônes, canalisations, réservoirs, points du réseau) sur l'emprise d'une entité du CSV, écrit chaque couche en GeoPackage dans `data/vector/reseaux/`, et affiche le résultat sur une carte.

In [ ]:
import sys
from pathlib import Path

# GeoDataEngine/ (racine du package core/) et son parent (racine du repo, contient data/)
project_root = Path.cwd().parent
repo_root = project_root.parent
sys.path.insert(0, str(project_root))

from core.config import load_entities
from download.limites_administratives import fetch_commune_boundary, resolve_bbox
from download.reseaux import download_reseaux

## 1. Entité testée (issue du CSV)

In [ ]:
csv_path = repo_root / "data" / "configuration" / "entites_exemple.csv"
entity = load_entities(csv_path)[0]
entity

## 2. Téléchargement des réseaux techniques sur la bbox de l'entité

In [ ]:
bbox = resolve_bbox(entity)
written = download_reseaux(bbox)
written

## 3. Relecture des GeoPackage écrits

In [ ]:
import geopandas as gpd

layers = {name: gpd.read_file(path) for name, path in written.items()}
for name, gdf in layers.items():
    print(name, "->", len(gdf), "entites")

## 4. Affichage sur une carte `leafmap`

Reprojection en EPSG:4326 uniquement pour l'affichage. Backend `folium` (HTML statique), plus fiable pour le rendu des widgets dans VS Code.

In [ ]:
import leafmap.foliumap as leafmap

styles = {
    "ligne_electrique": {"color": "red", "weight": 2},
    "poste_de_transformation": {"color": "darkred", "fillColor": "darkred", "fillOpacity": 0.6},
    "pylone": {"color": "orange", "fillColor": "orange", "fillOpacity": 0.6},
    "canalisation": {"color": "blue", "weight": 2},
    "reservoir": {"color": "teal", "fillColor": "teal", "fillOpacity": 0.5},
    "point_du_reseau": {"color": "purple", "fillColor": "purple", "fillOpacity": 0.6},
}

boundary_wgs84 = fetch_commune_boundary(entity.code_insee).to_crs(epsg=4326)

m = leafmap.Map()

for name, gdf in layers.items():
    if gdf.empty:
        continue
    m.add_gdf(
        gdf.to_crs(epsg=4326),
        layer_name=name,
        style=styles.get(name),
        info_mode="on_click",
    )

m.add_gdf(
    boundary_wgs84,
    layer_name="Limite communale",
    style={"color": "black", "fillOpacity": 0},
)
m.zoom_to_gdf(boundary_wgs84)
m